V10 Мат модель для нахождения коэффициента влияния гендера на ЗП

In [3]:
import pandas as pd

modelv1 = pd.read_excel("../../data/processed/v1_clean_base.xlsx")

model v10 (линейная регрессия по параметрам: salary, gender, experience, education_type, region_code, industry_code)

In [4]:
import pandas as pd
import statsmodels.formula.api as smf
import warnings

warnings.filterwarnings("ignore")

df = modelv1.copy()

df["salary"] = pd.to_numeric(df["salary"], errors="coerce")
df["experience"] = pd.to_numeric(df["experience"], errors="coerce").fillna(0)

df["region_code"] = (
    df["region_code"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D+", "", regex=True)
    .str.zfill(13)
    .str[:2]
)
df["region_code"] = pd.to_numeric(df["region_code"], errors="coerce").astype("Int64")

df = df[
    (df["salary"] > 5000) &
    (df["salary"] < 165000) &
    (df["gender"].isin(["Мужской", "Женский"])) &
    (df["industry_code"] == "BuldindRealty") &
    (df["region_code"].notna())
].copy()

df["male"] = (df["gender"] == "Мужской").astype(int)

results = []

for code, grp in df.groupby("region_code"):
    nm = int((grp["gender"] == "Мужской").sum())
    nf = int((grp["gender"] == "Женский").sum())

    if nm < 5 or nf < 5:
        continue

    reg = smf.ols("salary ~ male + experience", data=grp).fit()

    results.append({
        "регион": int(code),
        "n_М": nm,
        "n_Ж": nf,
        "n_всего": len(grp),
        "коэф_gender": round(reg.params["male"], 3),
        "p_gender": round(reg.pvalues["male"], 6),
        "gender_значимо": "да" if reg.pvalues["male"] < 0.05 else "нет",
        "коэф_опыт": round(reg.params["experience"], 3),
        "p_опыт": round(reg.pvalues["experience"], 6),
        "R2": round(reg.rsquared, 4)
    })

v10_result = pd.DataFrame(results).sort_values(["p_gender", "регион"]).reset_index(drop=True)

print(v10_result.to_string(index=False))


 регион  n_М  n_Ж  n_всего  коэф_gender  p_gender gender_значимо  коэф_опыт  p_опыт     R2
      0  597  430     1027     7450.034       0.0             да    641.014     0.0 0.1255


Сохранение результатов model v10

In [5]:
v10_result.to_excel("../../data/processed/v10_result.xlsx", index=False)